In [ ]:
#@title Environment Setup

import os
from pathlib import Path

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
MODELS_DIR = "/content/drive/MyDrive/LMStudio/models"

# Mount Google Drive for model persistence
if USE_GOOGLE_DRIVE:
    !echo "Mounting Google Drive..."
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(MODELS_DIR, exist_ok=True)
    print(f"Models will be stored in: {MODELS_DIR}")

# Install LM Studio only if not already installed
LMS_BIN = os.path.expanduser("~/.lmstudio/bin/lms")
if not os.path.exists(LMS_BIN):
    !echo -= Installing LM Studio =-
    !curl -fsSL https://lmstudio.ai/install.sh | bash
else:
    !echo -= LM Studio already installed, skipping =-

# Add lms to PATH
os.environ["PATH"] = os.path.expanduser("~/.lmstudio/bin") + ":" + os.environ["PATH"]

# Symlink BEFORE starting daemon so lms sees Drive models immediately
LMSTUDIO_MODELS = os.path.expanduser("~/.lmstudio/models")

if USE_GOOGLE_DRIVE:
    os.makedirs(os.path.expanduser("~/.lmstudio"), exist_ok=True)
    # Only re-symlink if not already pointing to Drive
    if not os.path.islink(LMSTUDIO_MODELS) or os.readlink(LMSTUDIO_MODELS) != MODELS_DIR:
        !rm -rf {LMSTUDIO_MODELS}
        !ln -sf {MODELS_DIR} {LMSTUDIO_MODELS}
    print(f"Symlinked {LMSTUDIO_MODELS} -> {MODELS_DIR}")
    print("Models will persist across sessions on Google Drive.")

# Start daemon after symlink so it picks up existing models
!lms daemon up

# Start server
!lms server start

Mounting Google Drive...


In [ ]:
#@title Expose via Pinggy Tunnel

import subprocess
import threading
import time
import socket

SERVER_PORT = 1234

def tunnel_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nLM Studio server ready, launching Pinggy tunnel...\n")

    p = subprocess.Popen(
        ["ssh", "-p", "443",
         "-R0:localhost:{}".format(port),
         "-o", "StrictHostKeyChecking=no",
         "-o", "ServerAliveInterval=45",
         "a@a.pinggy.io"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        stdin=subprocess.PIPE
    )
    for line in p.stdout:
        l = line.decode()
        if "pinggy.link" in l or "pinggy.io" in l:
            print("LM Studio API URL:", l.strip())

threading.Thread(target=tunnel_thread, daemon=True, args=(SERVER_PORT,)).start()

while True:
    time.sleep(60)